<a href="https://colab.research.google.com/github/NicolasPetiot/EnjeuxDecarbonationSante/blob/main/rosetta_mutate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
from pathlib import Path

if "google.colab" in sys.modules:

    from google.colab import drive
    drive.mount('/content/drive')

    DATA_DIR = Path("/content/drive/MyDrive/EnjeuxDecarbonationSanté/")

else:
    DATA_DIR = Path(".")


if not DATA_DIR.exists():
    raise FileNotFoundError(f"Le répertoire '{DATA_DIR}' n'est pas accessible ou n'existe pas")

In [ ]:
def rosetta_setup(verbosity = False, allow_overwrite = True, H_optimization = True, extra_params:list[Path] = []):
    flags = [
        f"-mute false" if verbosity else "-mute all",
        f"-ex1",
        f"-ex2",
        f"-no_optH {str(not H_optimization)}",
        f"-flip_HNQ true",
        f"-ignore_ligand_chi true",
        f"-overwrite" if allow_overwrite else "",
        f"-restore_pre_talaris_2013_behavior true"
    ]
    flags = " ".join(flags)

    extra_flag = []
    for path in extra_params:
        if not path.exists():
            print(f"WARNING: ignoring file {str(path)} (does not exists or is not acessible)")

        else:
            extra_flag.append(str(path))
    if len(extra_flag) > 0:
        flags += " -extra_res_fa " + " ".join(extra_flag)

    try:
        from pyrosetta import init

    except ImportError:
        !pip install pyrosetta_installer
        from pyrosetta_installer import install_pyrosetta
        install_pyrosetta()

        from pyrosetta import init

    finally:
        init(flags)

def load_rosetta_pose(path:Path):
    from pyrosetta import pose_from_file
    if path.exists():
        return pose_from_file(str(path))

    else:
        raise FileNotFoundError(f"Le fichier {str(path)} n'existe pas ou n'est pas accessible.")

In [ ]:
rosetta_setup(extra_params=[
    DATA_DIR / ".rosettafiles/GSH.params"
], verbosity=False, H_optimization=True)

In [ ]:
## Define Mutation Protocol:
from pyrosetta import Pose
from pyrosetta.rosetta.protocols.rosetta_scripts import XmlObjects

MUTATE_XML_STRING = """
<ROSETTASCRIPTS>
	<SCOREFXNS>
	</SCOREFXNS>
	<RESIDUE_SELECTORS>
        <Index name="to_mutate_A" resnums="{resi}A"/>
        <Index name="to_mutate_B" resnums="{resi}B"/>
        <Or name="to_mutate" selectors="to_mutate_A,to_mutate_B" />
	</RESIDUE_SELECTORS>
	<MOVERS>
        <MutateResidue name="mutate" residue_selector="to_mutate" new_res="{new_res}" preserve_atom_coords="false" />
	</MOVERS>
	<PROTOCOLS>
        <Add mover_name="mutate" />
	</PROTOCOLS>
	<OUTPUT />
</ROSETTASCRIPTS>
"""

def mutate(pose:Pose, aa_num:int, new_aa:str) -> Pose:
    """
    Apply a mutation to both chains A&B of an input pose.

    The mutation site is identified using an input residue index `resi:int`

    The new residue is specified using the single letter residue code `new_res`
    """
    xml = XmlObjects.create_from_string(MUTATE_XML_STRING.format(resi=aa_num, new_res=new_aa))
    protocol = xml.get_mover("ParsedProtocol")

    mutant = pose.clone()
    protocol.apply(mutant)
    return mutant

In [ ]:
## Define Binding Affinity Protocol:
from pyrosetta import Pose, ScoreFunction, Vector1
from pyrosetta import get_fa_scorefxn
from pyrosetta.rosetta import protocols

def binding_affinity(pose:Pose, partners = "AB_C", scorefxn:ScoreFunction = None) -> float:
    """
    Separates two partners and returns the difference of energy between bounded and separated states.
    """
    if scorefxn is None:
        scorefxn = get_fa_scorefxn()

    bind_score = scorefxn(pose)

    # Split partners:
    split_pose = pose.clone()
    jump = 2
    step_size = 100

    protocols.docking.setup_foldtree(pose, partners, Vector1([-1,-1,-1]))
    trans_mover = protocols.rigid.RigidBodyTransMover(split_pose,jump)
    trans_mover.step_size(step_size)
    trans_mover.apply(split_pose)

    split_score = scorefxn(split_pose)

    return bind_score - split_score

In [ ]:
pdb = DATA_DIR / "PDB/GSTD1+GSH.pdb"
pose = load_rosetta_pose(pdb)

dG = binding_affinity(pose)

# Choix de la mutation:
new_aa = "Met" # TODO: À modifier
aa_num = 1     # TODO: À modifier
save_mutant = False

mutant = mutate(pose, aa_num=aa_num, new_aa=new_aa.upper())
dG_mutant = binding_affinity(mutant)

if save_mutant:
    mutant_filename = str(pdb).replace(".pdb", f"_{new_aa}{aa_num}.pdb")
    mutant.dump_pdb(mutant_filename)

print(f"Mutation ddG: {dG_mutant - dG:.3f} R.E.U.")